# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ikramkhan-gif1/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# ML-10 setup

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

REPO_PATH = "/content/FlyRank-ML-Internship"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Ikramkhan-gif1/FlyRank-ML-Internship.git

DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")

Dataset shape: (30000, 44)
Loaded successfully.


In [15]:
# Recreate the Week-5 target

df["impression_change_pct"] = (
    (
        df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )
    / df["impressions_prev_30d"].replace(0, np.nan)
) * 100

df["is_declining_label"] = (
    df["impression_change_pct"] <= -20
).astype(int)

print("Target created successfully.")
print(df["is_declining_label"].value_counts())

Target created successfully.
is_declining_label
1    16305
0    13695
Name: count, dtype: int64


In [16]:
# Week-5 feature set

features = [
    "search_volume",
    "impressions_90d",
    "days_with_impressions",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "is_declining_label"

print("Features:", features)
print("Target:", target)

Features: ['search_volume', 'impressions_90d', 'days_with_impressions', 'impressions_last_30d', 'impressions_prev_30d', 'days_since_last_update', 'ctr', 'avg_position']
Target: is_declining_label


In [17]:
# Recreate the client-grouped validation used for the action queue

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        df,
        df[target],
        groups=df["client_id"]
    )
)

group_train = df.iloc[group_train_idx].copy()
group_test = df.iloc[group_test_idx].copy()

X_train_grouped = group_train[features].copy()
X_test_grouped = group_test[features].copy()

y_train_grouped = group_train[target].copy()
y_test_grouped = group_test[target].copy()

grouped_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_proba = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

print("Grouped model recreated successfully.")
print("Training rows:", len(group_train))
print("Testing rows:", len(group_test))

shared_clients = (
    set(group_train["client_id"])
    & set(group_test["client_id"])
)

print("Shared clients:", len(shared_clients))

Grouped model recreated successfully.
Training rows: 23837
Testing rows: 6163
Shared clients: 0


## 1. Ranked actions + reason codes
### Ranked actions

The action queue ranks content items by the measured decision-support signal available from the validated model output.

The purpose of the ranking is to help a human reviewer prioritize which content items deserve attention first. The ranking is not an automatic instruction to change content.

Each recommendation includes a reason code so that a reviewer can understand why the item was placed in the queue. The main reason codes are based on observed decline, content freshness, and the strength of the model's measured signal.

### Reason codes

* **DECLINE_SIGNAL** — the observed data indicate a decline according to the defined proxy label or model signal.
* **STALE_CONTENT** — the content has a high `days_since_last_update` value and may deserve freshness review.
* **STRONG_SIGNAL** — the model assigns a relatively high probability to the declining class.
* **COMBINED_RISK** — multiple signals indicate that the item may deserve earlier human review.

The highest-ranked items should receive human review first. The queue is intended for prioritization and decision-support, not automatic publishing or automatic content changes.


In [18]:
# ML-10 — Build ranked action queue

# Use the grouped-model output created in ML-09.
# If the notebook is run independently, these variables must already
# exist from the earlier validation/modeling cells.

action_queue = group_test.copy()

# Add model probability
action_queue["decline_probability"] = grouped_proba

# Create model signal
action_queue["model_signal"] = (
    action_queue["decline_probability"] >= 0.50
).astype(int)

# Create reason components
action_queue["reason_stale"] = (
    action_queue["days_since_last_update"] >= 91
)

action_queue["reason_decline"] = (
    action_queue["is_declining_label"] == 1
)

action_queue["reason_strong_signal"] = (
    action_queue["decline_probability"] >= 0.75
)

# Build human-readable reason codes
def create_reason_code(row):
    reasons = []

    if row["reason_decline"]:
        reasons.append("DECLINE_SIGNAL")

    if row["reason_stale"]:
        reasons.append("STALE_CONTENT")

    if row["reason_strong_signal"]:
        reasons.append("STRONG_SIGNAL")

    if len(reasons) >= 2:
        return "COMBINED_RISK"

    if len(reasons) == 1:
        return reasons[0]

    return "MONITOR"


action_queue["reason_code"] = action_queue.apply(
    create_reason_code,
    axis=1
)

# Assign a practical human-review action
def assign_action(row):
    if row["reason_code"] == "COMBINED_RISK":
        return "REVIEW_FIRST"

    if row["reason_code"] in [
        "DECLINE_SIGNAL",
        "STRONG_SIGNAL"
    ]:
        return "REVIEW"

    if row["reason_code"] == "STALE_CONTENT":
        return "FRESHNESS_REVIEW"

    return "MONITOR"


action_queue["recommended_action"] = action_queue.apply(
    assign_action,
    axis=1
)

# Rank the queue using model signal and staleness
action_queue["priority_score"] = (
    action_queue["decline_probability"] * 100
    + action_queue["days_since_last_update"].clip(upper=365) / 365 * 20
)

action_queue = action_queue.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = (
    action_queue.index + 1
)

print("Ranked action queue created.")
print("Rows:", len(action_queue))

action_queue[
    [
        "rank",
        "recommended_action",
        "reason_code",
        "decline_probability",
        "days_since_last_update",
        "priority_score"
    ]
].head(20)


Ranked action queue created.
Rows: 6163


,rank,recommended_action,reason_code,decline_probability,days_since_last_update,priority_score
0,1,REVIEW_FIRST,COMBINED_RISK,1.000000,106,105.808219
1,2,REVIEW_FIRST,COMBINED_RISK,1.000000,106,105.808219
2,3,REVIEW_FIRST,COMBINED_RISK,0.999990,106,105.807205
3,4,REVIEW_FIRST,COMBINED_RISK,0.999433,106,105.751542
4,5,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
5,6,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
6,7,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
7,8,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
8,9,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698626
9,10,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698621


## 2. Intended use and limits

### Intended use

The action playbook is intended for analysts, SEO practitioners, or content teams who need to prioritize content items for human review.

The ranked queue is a decision-support tool. It helps a reviewer identify content items that show stronger measured signals of decline, freshness concerns, or both.

The queue should be used to decide **what to review first**, not to decide automatically **what content should be changed**.

### Limits

The results are based on the evaluated dataset and the defined `is_declining_label`. They should not be treated as a guarantee of future performance.

The model provides directional decision-support signal rather than a production-ready automated recommendation system.

The validation audit also identified target-construction overlap because `impressions_last_30d` and `impressions_prev_30d` are both used to construct the label and included as model features. This limits how strongly the measured model score can be interpreted.

The playbook therefore supports prioritization and human investigation, but it does not establish that a content change will cause traffic or impressions to improve.


In [19]:
# Summarize the current action queue

action_summary = (
    action_queue["recommended_action"]
    .value_counts()
    .rename_axis("Recommended action")
    .reset_index(name="Number of items")
)

action_summary


,Recommended action,Number of items
0,MONITOR,2348
1,REVIEW,1804
2,REVIEW_FIRST,1363
3,FRESHNESS_REVIEW,648


## 3. Human review + the no-go list
### Human review rules

Every item in the action queue should be reviewed by a human before any content change is made.

The reviewer should check:

1. Whether the observed decline signal is meaningful and not caused by a short-term fluctuation.
2. Whether the content is still relevant to the search intent.
3. Whether the page has outdated, incomplete, or inaccurate information.
4. Whether the recommendation is consistent with the actual content and business context.
5. Whether there are technical, seasonal, or external reasons that could explain the observed change.

### No-go cases

The system should **not** automatically:

- publish or edit content;
- delete or redirect pages;
- change titles, metadata, or internal links;
- change content based only on model probability;
- claim that a refresh will improve traffic or rankings;
- make decisions involving sensitive business or client information;
- replace expert review for high-impact content decisions.

The model output is therefore limited to prioritization and decision-support. A human remains responsible for the final action.

In [20]:
# Human-review categories in the current queue

review_rules = pd.DataFrame({
    "Recommended action": [
        "REVIEW_FIRST",
        "REVIEW",
        "FRESHNESS_REVIEW",
        "MONITOR"
    ],
    "Human review required": [
        "Yes",
        "Yes",
        "Yes",
        "Optional monitoring"
    ],
    "Purpose": [
        "Prioritize strongest combined signals",
        "Investigate measured decline or strong model signal",
        "Check whether content may be outdated",
        "Watch for future changes"
    ]
})

review_rules


,Recommended action,Human review required,Purpose
0,REVIEW_FIRST,Yes,Prioritize strongest combined signals
1,REVIEW,Yes,Investigate measured decline or strong model s...
2,FRESHNESS_REVIEW,Yes,Check whether content may be outdated
3,MONITOR,Optional monitoring,Watch for future changes


## 4. Monitoring / retrain triggers

### Monitoring and retrain triggers

The action playbook should be monitored periodically to check whether its measured signals remain useful.

The following indicators should be reviewed:

- **Model performance:** Recalculate ROC-AUC on a later labeled sample when labels become available.
- **Prediction distribution:** Monitor whether the distribution of decline probabilities changes substantially.
- **Feature drift:** Check whether important input features such as impressions, CTR, position, or freshness change substantially from the development data.
- **Action distribution:** Monitor whether the queue suddenly contains an unusually high or low proportion of `REVIEW_FIRST` or `REVIEW` recommendations.
- **Data quality:** Check for missing values, unexpected ranges, duplicated observations, or changes in feature definitions.

### Retrain triggers

A model review or retraining investigation should be considered when:

1. Measured ROC-AUC decreases materially on a later evaluation sample.
2. Input feature distributions show sustained drift.
3. The distribution of recommended actions changes substantially.
4. The target definition or source data changes.
5. New evidence shows that the current features no longer provide useful directional signal.

These are review triggers rather than automatic retraining rules. A human should investigate the cause before deciding whether retraining is appropriate.

In [21]:
# Monitoring summary for the current action queue

monitoring_summary = pd.DataFrame({
    "Metric": [
        "Queue size",
        "REVIEW_FIRST items",
        "REVIEW items",
        "FRESHNESS_REVIEW items",
        "MONITOR items",
        "Mean decline probability",
        "Median decline probability"
    ],
    "Current value": [
        len(action_queue),
        (action_queue["recommended_action"] == "REVIEW_FIRST").sum(),
        (action_queue["recommended_action"] == "REVIEW").sum(),
        (action_queue["recommended_action"] == "FRESHNESS_REVIEW").sum(),
        (action_queue["recommended_action"] == "MONITOR").sum(),
        round(action_queue["decline_probability"].mean(), 4),
        round(action_queue["decline_probability"].median(), 4)
    ]
})

monitoring_summary


,Metric,Current value
0,Queue size,6163.0000
1,REVIEW_FIRST items,1363.0000
2,REVIEW items,1804.0000
3,FRESHNESS_REVIEW items,648.0000
4,MONITOR items,2348.0000
5,Mean decline probability,0.4855
6,Median decline probability,0.4419


## 5. Exports for the paper

### Exports for the paper

The action queue is exported so that the paper can reuse the measured recommendations without manually copying results from the notebook.

The exported queue contains the ranking, recommended action, reason code, model probability, and relevant freshness signal.

The queue is an analysis output and is regenerated by the notebook rather than treated as a production data feed.

The export directory is:

`work/outputs/`

The CSV is intentionally kept outside the committed repository data files according to the assignment instructions.

In [22]:
# Create the required output directory

OUTPUT_DIR = os.path.join(
    REPO_PATH,
    "work",
    "outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Output directory:", OUTPUT_DIR)

Output directory: /content/FlyRank-ML-Internship/work/outputs


In [23]:
# Export the ranked action queue

queue_columns = [
    "rank",
    "recommended_action",
    "reason_code",
    "decline_probability",
    "days_since_last_update",
    "priority_score"
]

queue_export = action_queue[
    [col for col in queue_columns if col in action_queue.columns]
].copy()

queue_path = os.path.join(
    OUTPUT_DIR,
    "ml10_ranked_action_queue.csv"
)

queue_export.to_csv(
    queue_path,
    index=False
)

print("Queue exported successfully:")
print(queue_path)
print("Rows exported:", len(queue_export))

Queue exported successfully:
/content/FlyRank-ML-Internship/work/outputs/ml10_ranked_action_queue.csv
Rows exported: 6163


In [24]:
# Verify that the exported queue can be read back

verified_queue = pd.read_csv(queue_path)

print("Export verified.")
print("Shape:", verified_queue.shape)

verified_queue.head(10)

Export verified.
Shape: (6163, 6)


,rank,recommended_action,reason_code,decline_probability,days_since_last_update,priority_score
0,1,REVIEW_FIRST,COMBINED_RISK,1.000000,106,105.808219
1,2,REVIEW_FIRST,COMBINED_RISK,1.000000,106,105.808219
2,3,REVIEW_FIRST,COMBINED_RISK,0.999990,106,105.807205
3,4,REVIEW_FIRST,COMBINED_RISK,0.999433,106,105.751542
4,5,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
5,6,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
6,7,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
7,8,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698630
8,9,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698626
9,10,REVIEW_FIRST,COMBINED_RISK,1.000000,104,105.698621


### Final audit note

This notebook converts the validated model output into a ranked content action playbook for human review.

The queue provides directional decision-support based on the measured model signal, observed freshness, and the defined decline proxy. It is intended to help prioritize review rather than automatically make content changes.

The playbook includes human-review rules, explicit no-go cases, monitoring and retrain triggers, and an export that can be reused by the research paper.

The recommendations should not be interpreted as proof that a content refresh will improve traffic, impressions, rankings, or other outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.